In [1]:
import pandas as pd

In [2]:
data=pd.read_csv("/content/drive/MyDrive/URL dataset.csv")
data

,url,type
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate
...,...,...
450171,http://ecct-it.com/docmmmnn/aptgd/index.php,phishing
450172,http://faboleena.com/js/infortis/jquery/plugin...,phishing
450173,http://faboleena.com/js/infortis/jquery/plugin...,phishing
450174,http://atualizapj.com/,phishing


In [3]:
!pip install -q xgboost lightgbm scikit-learn pandas numpy joblib tqdm

In [4]:
import warnings
warnings.filterwarnings('ignore')

import re, math, json, joblib
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from collections import Counter
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    matthews_corrcoef, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = Path('phishing_output')
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'models').mkdir(exist_ok=True)
print('Imports ready')

Imports ready


In [5]:
# ── Known legitimate domains (Fix B) ─────────────────────────────────────
KNOWN_LEGIT_DOMAINS = {
    # Search engines
    'google.com', 'google.co.uk', 'google.com.au', 'google.ca',
    'bing.com', 'yahoo.com', 'duckduckgo.com', 'baidu.com',
    # Developer / tech
    'github.com', 'gitlab.com', 'stackoverflow.com', 'stackexchange.com',
    'reddit.com', 'medium.com', 'dev.to', 'npmjs.com', 'pypi.org',
    'readthedocs.io', 'docs.python.org', 'mozilla.org',
    # Major news / media
    'bbc.co.uk', 'bbc.com', 'cnn.com', 'nytimes.com', 'theguardian.com',
    'reuters.com', 'apnews.com', 'washingtonpost.com', 'forbes.com',
    # E-commerce / services
    'amazon.com', 'amazon.co.uk', 'ebay.com', 'etsy.com', 'shopify.com',
    'paypal.com', 'stripe.com', 'apple.com', 'microsoft.com',
    # Social
    'facebook.com', 'instagram.com', 'twitter.com', 'x.com',
    'linkedin.com', 'youtube.com', 'tiktok.com', 'pinterest.com',
    # Cloud / infra
    'aws.amazon.com', 'azure.microsoft.com', 'cloud.google.com',
    'cloudflare.com', 'digitalocean.com', 'heroku.com',
    # Education / gov
    'wikipedia.org', 'wikimedia.org', 'archive.org',
    'gov.uk', 'usa.gov', 'whitehouse.gov', 'europa.eu',
    # Misc well-known
    'example.com', 'w3schools.com', 'coursera.org', 'udemy.com',
    'netflix.com', 'spotify.com', 'dropbox.com', 'box.com',
}

LEGIT_TLDS = {
    'com', 'org', 'net', 'edu', 'gov', 'co', 'io', 'uk', 'de',
    'fr', 'jp', 'au', 'ca', 'mil', 'int', 'eu', 'us', 'info'
}

SUSPICIOUS_TLDS = {
    'xyz', 'top', 'club', 'online', 'site', 'website', 'space',
    'fun', 'live', 'click', 'link', 'download', 'win', 'bid',
    'trade', 'gq', 'ml', 'cf', 'tk', 'ga', 'men', 'loan'
}

# Kept only words that are inherently phishing-specific.
PHISHING_KEYWORDS = [
    'verify', 'signin',
    'banking', 'payment', 'wallet',
    'password', 'recovery',
    'helpdesk', 'refund',
    'prize', 'winner', 'claim',
    'crypto', 'bitcoin',
    'webscr', 'ebayisapi',
    'cmd=_login', 'dispatch',
]

BRAND_NAMES = [
    'paypal', 'ebay', 'amazon', 'google', 'microsoft',
    'apple', 'facebook', 'instagram', 'netflix', 'bankofamerica',
    'wellsfargo', 'chase', 'citibank',
]

URL_SHORTENERS = {
    'bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly',
    'buff.ly', 'adf.ly', 'short.link', 'tiny.cc', 'is.gd'
}


def shannon_entropy(s):
    if not s:
        return 0.0
    counts = Counter(s)
    total = len(s)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def is_ip_address(host):
    ipv4 = re.match(r'^(\d{1,3}\.){3}\d{1,3}$', host)
    ipv6 = re.match(r'^\[?[0-9a-fA-F:]+\]?$', host)
    return int(bool(ipv4 or ipv6))


def count_subdomains(host):
    parts = host.split('.')
    return max(0, len(parts) - 2)


def brand_used_suspiciously(url_lower, domain):
    """Returns 1 only when a brand is spoofed (not when it IS the real domain)."""
    for brand in BRAND_NAMES:
        if brand not in url_lower:
            continue
        if domain.startswith(brand + '.'):
            return 0   # real brand domain -- not suspicious
        return 1       # brand in subdomain/path of a foreign domain -- spoofing
    return 0


def extract_features(url):
    """Extract all real-time computable features from a raw URL string.
    This exact function is used at both training AND inference time.
    Returns a flat dict of numeric features.

    Applied fixes:
      Fix A: slash_path_ratio  -- n_slashes normalised by url_length (not raw count)
      Fix B: is_known_legit_domain -- strong protective anchor for well-known domains
      Fix C: risk_score removed from returned dict (not passed to model)
    """
    url = str(url).strip()
    if not re.match(r'^[a-zA-Z][a-zA-Z0-9+\-.]*://', url):
        url_to_parse = 'http://' + url
    else:
        url_to_parse = url
    try:
        parsed   = urlparse(url_to_parse)
        scheme   = parsed.scheme.lower()
        host     = parsed.hostname or ''
        port     = parsed.port
        path     = parsed.path or ''
        query    = parsed.query or ''
    except Exception:
        scheme = host = path = query = ''
        port = None

    eps        = 1e-9
    url_lower  = url.lower()
    url_length = len(url)

    n_digits      = sum(c.isdigit() for c in url)
    n_letters     = sum(c.isalpha() for c in url)
    n_dots        = url.count('.')
    n_hyphens     = url.count('-')
    n_underscores = url.count('_')
    # Count slashes in path only, NOT the '//' inside 'https://'
    n_slashes     = path.count('/')
    n_equals      = url.count('=')
    n_qmarks      = url.count('?')
    n_amps        = url.count('&')
    n_at          = url.count('@')
    n_percent     = url.count('%')

    host_parts = host.split('.')
    tld    = host_parts[-1].lower() if host_parts else ''
    domain = '.'.join(host_parts[-2:]) if len(host_parts) >= 2 else host

    try:
        params = parse_qs(query)
        n_params = len(params)
        param_values = [str(v[0]) for v in params.values()]
    except Exception:
        n_params = 0
        param_values = []

    is_https         = int(scheme == 'https')
    is_http          = int(scheme == 'http')
    has_non_std_port = int(port is not None and port not in (80, 443, 8080, 8443))

    domain_length       = len(domain)
    host_length         = len(host)
    n_subdomains        = count_subdomains(host)
    is_ip               = is_ip_address(host)
    domain_entropy      = shannon_entropy(domain)
    host_entropy        = shannon_entropy(host)
    n_dots_in_host      = host.count('.')
    n_hyphens_in_domain = domain.count('-')
    domain_digit_ratio  = sum(c.isdigit() for c in domain) / (domain_length + eps)
    is_shortener        = int(any(s in host for s in URL_SHORTENERS))
    has_dash_in_domain  = int('-' in (domain.split('.')[0] if '.' in domain else domain))
    multiple_subdomains = int(n_subdomains >= 3)

    tld_length        = len(tld)
    is_legit_tld      = int(tld in LEGIT_TLDS)
    is_suspicious_tld = int(tld in SUSPICIOUS_TLDS)

    path_length   = len(path)
    n_path_dirs   = max(0, path.count('/') - 1)
    path_entropy  = shannon_entropy(path)
    n_path_digits = sum(c.isdigit() for c in path)
    path_has_exe  = int(bool(re.search(r'\.(exe|bat|sh|cmd|msi|vbs|ps1)$', path.lower())))
    path_has_form = int(any(kw in path.lower() for kw in ('signin', 'verify')))

    query_length      = len(query)
    n_query_params    = n_params
    query_entropy     = shannon_entropy(query)
    avg_param_val_len = float(np.mean([len(v) for v in param_values])) if param_values else 0.0

    has_at_symbol    = int(n_at > 0)
    has_double_slash = int(path.count('//') > 0)
    has_hex_chars    = int(bool(re.search(r'%[0-9a-fA-F]{2}', url)))
    hex_char_count   = len(re.findall(r'%[0-9a-fA-F]{2}', url))
    has_punycode     = int('xn--' in host.lower())

    n_phishing_keywords  = sum(1 for kw in PHISHING_KEYWORDS if kw in url_lower)
    has_phishing_keyword = int(n_phishing_keywords > 0)
    has_brand_name       = brand_used_suspiciously(url_lower, domain)

    digit_ratio        = n_digits  / (url_length + eps)
    letter_ratio       = n_letters / (url_length + eps)
    special_char_ratio = (n_dots + n_hyphens + n_underscores +
                          n_equals + n_qmarks + n_amps + n_at + n_percent) / (url_length + eps)
    url_entropy        = shannon_entropy(url)
    domain_url_ratio   = domain_length / (url_length + eps)
    path_url_ratio     = path_length   / (url_length + eps)

    # Fix A: slash_path_ratio — normalise slashes by URL length.
    # A long legitimate URL (github.com/user/repo) gets a LOW ratio;
    # a short URL with many slashes gets a HIGH ratio — more meaningful.
    slash_path_ratio = n_slashes / (url_length + eps)

    # Fix B: is_known_legit_domain — strong protective anchor.
    # Checks both the registered domain (e.g. github.com) and
    # common subdomains (e.g. accounts.google.com -> google.com).
    is_known_legit_domain = int(
        domain in KNOWN_LEGIT_DOMAINS or
        host in KNOWN_LEGIT_DOMAINS or
        any(host.endswith('.' + d) for d in KNOWN_LEGIT_DOMAINS)
    )

    # Fix C: risk_score is NOT returned — it is kept here only for the
    # sanity-check print below and is excluded from FEATURE_NAMES so it
    # is never passed to the model.  This prevents the model from
    # double-counting signals that are already individual features.
    risk_score = (
        is_ip               * 3.0 +
        has_at_symbol       * 2.0 +
        has_hex_chars       * 1.0 +
        is_suspicious_tld   * 2.5 +
        multiple_subdomains * 1.5 +
        (1 - is_https)      * 0.5 +
        has_punycode        * 2.0 +
        n_phishing_keywords * 1.5 +
        has_brand_name      * 2.0 +
        is_shortener        * 1.5 +
        is_legit_tld        * -1.5 +
        is_known_legit_domain * -3.0   # strong negative weight
    )
    risk_score = max(0.0, risk_score)

    return {
        'url_length':              url_length,
        'url_entropy':             round(url_entropy, 4),
        'digit_ratio':             round(digit_ratio, 4),
        'letter_ratio':            round(letter_ratio, 4),
        'special_char_ratio':      round(special_char_ratio, 4),
        'n_dots':                  n_dots,
        'n_hyphens':               n_hyphens,
        'n_underscores':           n_underscores,
        # Fix A: replaced raw n_slashes with normalised slash_path_ratio
        'slash_path_ratio':        round(slash_path_ratio, 6),
        'n_equals':                n_equals,
        'n_qmarks':                n_qmarks,
        'n_amps':                  n_amps,
        'n_at':                    n_at,
        'n_percent':               n_percent,
        'n_digits':                n_digits,
        'n_letters':               n_letters,
        'is_https':                is_https,
        'is_http':                 is_http,
        'has_non_std_port':        has_non_std_port,
        'domain_length':           domain_length,
        'host_length':             host_length,
        'n_subdomains':            n_subdomains,
        'is_ip':                   is_ip,
        'domain_entropy':          round(domain_entropy, 4),
        'host_entropy':            round(host_entropy, 4),
        'n_dots_in_host':          n_dots_in_host,
        'n_hyphens_in_domain':     n_hyphens_in_domain,
        'domain_digit_ratio':      round(domain_digit_ratio, 4),
        'is_shortener':            is_shortener,
        'has_dash_in_domain':      has_dash_in_domain,
        'multiple_subdomains':     multiple_subdomains,
        'tld_length':              tld_length,
        'is_legit_tld':            is_legit_tld,
        'is_suspicious_tld':       is_suspicious_tld,
        'path_length':             path_length,
        'n_path_dirs':             n_path_dirs,
        'path_entropy':            round(path_entropy, 4),
        'n_path_digits':           n_path_digits,
        'path_has_exe':            path_has_exe,
        'path_has_form':           path_has_form,
        'path_url_ratio':          round(path_url_ratio, 4),
        'query_length':            query_length,
        'n_query_params':          n_query_params,
        'query_entropy':           round(query_entropy, 4),
        'avg_param_val_len':       round(avg_param_val_len, 4),
        'has_at_symbol':           has_at_symbol,
        'has_double_slash':        has_double_slash,
        'has_hex_chars':           has_hex_chars,
        'hex_char_count':          hex_char_count,
        'has_punycode':            has_punycode,
        'n_phishing_keywords':     n_phishing_keywords,
        'has_phishing_keyword':    has_phishing_keyword,
        'has_brand_name':          has_brand_name,
        'domain_url_ratio':        round(domain_url_ratio, 4),
        # Fix B: new protective feature
        'is_known_legit_domain':   is_known_legit_domain,
        # Fix C: risk_score REMOVED from returned dict
        #        (kept as local variable above for the sanity-check only)
    }


FEATURE_NAMES = list(extract_features('http://example.com').keys())

# Sanity check -- legit URLs must show LOW risk, phishing must show HIGH risk
print('=== Sanity check (v3: Fix A+B+C applied) ===')
test_cases = [
    ('https://www.google.com',                               'LEGIT'),
    ('https://github.com/user/repo',                         'LEGIT'),
    ('https://accounts.google.com/signin/v2/identifier',     'LEGIT'),
    ('https://www.amazon.com/dp/B08N5WRWNW',                 'LEGIT'),
    ('https://support.apple.com/en-us/HT213890',             'LEGIT'),
    ('http://www.bbc.co.uk/news',                            'LEGIT'),
    ('http://192.168.1.1/login/verify?user=admin',           'PHISH'),
    ('http://paypal-secure-login.xyz/account/verify',        'PHISH'),
    ('http://www.amazon.com.phishing.tk/signin',             'PHISH'),
    ('https://secure-bankofamerica.verify-account.ml/login', 'PHISH'),
]
print(f"{'Expected':<8} {'legit_dom':>9}  {'https':>5}  {'kw':>3}  {'brand':>5}  {'slash_ratio':>11}  URL")
print('-' * 90)
for url, expected in test_cases:
    f = extract_features(url)
    print(f"{expected:<8} {f['is_known_legit_domain']:>9}  {f['is_https']:>5}  "
          f"{f['n_phishing_keywords']:>3}  {f['has_brand_name']:>5}  "
          f"{f['slash_path_ratio']:>11.5f}  {url[:55]}")

print(f'\nTotal features: {len(FEATURE_NAMES)}')
print(f'Features: {FEATURE_NAMES}')


=== Sanity check (v3: Fix A+B+C applied) ===
Expected legit_dom  https   kw  brand  slash_ratio  URL
------------------------------------------------------------------------------------------
LEGIT            1      1    0      0      0.00000  https://www.google.com
LEGIT            1      1    0      0      0.07143  https://github.com/user/repo
LEGIT            1      1    1      0      0.06250  https://accounts.google.com/signin/v2/identifier
LEGIT            1      1    0      0      0.05556  https://www.amazon.com/dp/B08N5WRWNW
LEGIT            1      1    0      0      0.05000  https://support.apple.com/en-us/HT213890
LEGIT            1      0    0      0      0.04000  http://www.bbc.co.uk/news
PHISH            0      0    1      0      0.04762  http://192.168.1.1/login/verify?user=admin
PHISH            0      0    1      1      0.04444  http://paypal-secure-login.xyz/account/verify
PHISH            0      0    1      1      0.02500  http://www.amazon.com.phishing.tk/signin
PHISH

In [6]:
# ── Load the new URL dataset ────────────────────────────────────────────────
data = pd.read_csv("/content/drive/MyDrive/URL dataset.csv")

print(f'Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')
print(f'\nType distribution:')
print(data.iloc[:, -1].value_counts())   # preview last column (likely the label)
print(f'\nFirst 3 rows:')
print(data.head(3))

Shape: (450176, 2)
Columns: ['url', 'type']

Type distribution:
type
legitimate    345738
phishing      104438
Name: count, dtype: int64

First 3 rows:
                        url        type
0    https://www.google.com  legitimate
1   https://www.youtube.com  legitimate
2  https://www.facebook.com  legitimate


In [7]:
# ── Detect URL and label columns ────────────────────────────────────────────
# Based on your sample: columns are likely 'url' and 'type'
URL_COL   = 'url'
LABEL_COL = 'type'

df_work = data[[URL_COL, LABEL_COL]].dropna().copy()
df_work.columns = ['url', 'label']

# Map text labels to 0=legit, 1=malicious
LABEL_MAP = {
    'legitimate': 0,
    'benign':     0,
    'phishing':   1,
    'malware':    1,
    'defacement': 1,
}
df_work['label'] = df_work['label'].str.strip().str.lower().map(LABEL_MAP)

# Catch any unmapped values
unmapped = df_work['label'].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} unmapped rows — check these label values:")
    print(data[LABEL_COL][df_work['label'].isna()].value_counts())

df_work = df_work.dropna(subset=['label'])
df_work['label'] = df_work['label'].astype(int)

print(f'Final shape: {df_work.shape}')
print(f'\nLabel distribution (0=legit, 1=malicious):')
print(df_work['label'].value_counts())
print(f'\nSample LEGIT URLs:')
print(df_work[df_work['label']==0]['url'].head(5).tolist())
print(f'\nSample PHISHING URLs:')
print(df_work[df_work['label']==1]['url'].head(5).tolist())

Final shape: (450176, 2)

Label distribution (0=legit, 1=malicious):
label
0    345738
1    104438
Name: count, dtype: int64

Sample LEGIT URLs:
['https://www.google.com', 'https://www.youtube.com', 'https://www.facebook.com', 'https://www.baidu.com', 'https://www.wikipedia.org']

Sample PHISHING URLs:
['http://atualizacaodedados.online', 'http://webmasteradmin.ukit.me/', 'http://stcdxmt.bigperl.in/klxtv/apps/uk/', 'https://tubuh-syarikat.com/plugins/fields/files/', 'http://rolyborgesmd.com/exceword/excel.php?.rand=13InboxLight.aspx?n==']


In [8]:
# ── Apply feature extractor to every URL ────────────────────────────────────
print(f'Extracting features from {len(df_work):,} URLs...')
feat_list = []
for url in tqdm(df_work['url'], desc='Extracting features'):
    feat_list.append(extract_features(url))

X = pd.DataFrame(feat_list, index=df_work.index)
y = df_work['label']

print(f'Feature matrix shape : {X.shape}')
print(f'Phishing rate        : {y.mean():.3f}')

# Sanity check — make sure legit URLs now have real path variety
print(f'\nFeature means by class:')
print(X.join(y).groupby('label')[['url_length','slash_path_ratio','n_path_dirs','is_https']].mean().round(3))

Extracting features from 450,176 URLs...


Extracting features: 100%|██████████| 450176/450176 [00:56<00:00, 7970.62it/s]


Feature matrix shape : (450176, 55)
Phishing rate        : 0.232

Feature means by class:
       url_length  slash_path_ratio  n_path_dirs  is_https
label                                                     
0          58.481             0.040        1.298     1.000
1          66.052             0.049        1.859     0.062


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

BINARY_FEATURES = [
    'is_https', 'is_http', 'is_ip', 'is_shortener', 'is_legit_tld',
    'is_suspicious_tld', 'has_at_symbol', 'has_double_slash', 'has_hex_chars',
    'has_punycode', 'has_phishing_keyword', 'has_brand_name', 'path_has_exe',
    'path_has_form', 'has_dash_in_domain', 'multiple_subdomains',
    'has_non_std_port', 'is_known_legit_domain',
]
CONTINUOUS_FEATURES = [f for f in FEATURE_NAMES if f not in BINARY_FEATURES]

print(f'Continuous features : {len(CONTINUOUS_FEATURES)}')
print(f'Binary features     : {len(BINARY_FEATURES)}')

scaler = RobustScaler()
X_train_sc = X_train.copy()
X_test_sc  = X_test.copy()
X_train_sc[CONTINUOUS_FEATURES] = scaler.fit_transform(X_train[CONTINUOUS_FEATURES])
X_test_sc[CONTINUOUS_FEATURES]  = scaler.transform(X_test[CONTINUOUS_FEATURES])
print('Scaling done')

Train: (360140, 55)  |  Test: (90036, 55)
Continuous features : 37
Binary features     : 18
Scaling done


In [10]:
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        eval_metric='logloss', tree_method='hist',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1
    ),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_sc, y_train)
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    metrics = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'mcc':       matthews_corrcoef(y_test, y_pred),
    }
    results[name] = {'model': model, 'metrics': metrics, 'y_pred': y_pred, 'y_proba': y_proba}
    print(f'  Accuracy={metrics["accuracy"]:.4f}  F1={metrics["f1"]:.4f}  '
          f'ROC-AUC={metrics["roc_auc"]:.4f}  MCC={metrics["mcc"]:.4f}')

print('All models trained.')

Training RandomForest...
  Accuracy=0.9967  F1=0.9929  ROC-AUC=0.9993  MCC=0.9908
Training XGBoost...
  Accuracy=0.9973  F1=0.9943  ROC-AUC=0.9994  MCC=0.9925
Training LightGBM...
  Accuracy=0.9973  F1=0.9942  ROC-AUC=0.9993  MCC=0.9925
All models trained.


In [11]:
best_name  = max(results, key=lambda n: results[n]['metrics']['f1'])
best_model = results[best_name]['model']
y_proba_best = results[best_name]['y_proba']

print(f'Best model: {best_name}')
df_metrics = pd.DataFrame({n: r['metrics'] for n, r in results.items()}).T
print(df_metrics.round(4))

Best model: XGBoost
              accuracy  precision  recall      f1  roc_auc     mcc
RandomForest    0.9967     0.9983  0.9876  0.9929   0.9993  0.9908
XGBoost         0.9973     0.9965  0.9920  0.9943   0.9994  0.9925
LightGBM        0.9973     0.9966  0.9919  0.9942   0.9993  0.9925


In [12]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_best)
f1_vals  = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = f1_vals.argmax()
_auto_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5

# Let the data decide — don't hardcode 0.65 anymore
# The new dataset has real paths in legit URLs so the model won't be biased
FINAL_THRESHOLD = _auto_threshold

print(f'Auto-optimal threshold : {_auto_threshold:.3f}')
yopt = (y_proba_best >= FINAL_THRESHOLD).astype(int)
print(f'  F1={f1_score(y_test, yopt):.4f}  Recall={recall_score(y_test, yopt):.4f}')
print(f'\nFinal threshold: {FINAL_THRESHOLD:.3f}')

Auto-optimal threshold : 0.549
  F1=0.9944  Recall=0.9918

Final threshold: 0.549


In [13]:
safe_name   = best_name.replace(' ', '_')
model_path  = OUT_DIR / 'models' / f'{safe_name}.pkl'
scaler_path = OUT_DIR / 'models' / 'robust_scaler.pkl'

joblib.dump(best_model, model_path)
joblib.dump(scaler,     scaler_path)

config = {
    'best_model':          best_name,
    'model_path':          str(model_path),
    'scaler_path':         str(scaler_path),
    'feature_names':       FEATURE_NAMES,
    'binary_features':     BINARY_FEATURES,
    'continuous_features': CONTINUOUS_FEATURES,
    'threshold':           FINAL_THRESHOLD,
    'test_metrics':        {k: round(v, 4) for k, v in results[best_name]['metrics'].items()},
}
with open(OUT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Model  saved : {model_path}')
print(f'Scaler saved : {scaler_path}')
print(f'Config saved : {OUT_DIR}/config.json')

Model  saved : phishing_output/models/XGBoost.pkl
Scaler saved : phishing_output/models/robust_scaler.pkl
Config saved : phishing_output/config.json


In [23]:
def predict_url(url, model=None, scaler_obj=None, threshold=None):
    if model is None:      model      = joblib.load(model_path)
    if scaler_obj is None: scaler_obj = joblib.load(scaler_path)
    if threshold is None:  threshold  = FINAL_THRESHOLD

    feats    = extract_features(url)
    X_input  = pd.DataFrame([feats])
    X_scaled = X_input.copy()
    X_scaled[CONTINUOUS_FEATURES] = scaler_obj.transform(X_input[CONTINUOUS_FEATURES])

    proba_phish = model.predict_proba(X_scaled[FEATURE_NAMES])[0][1]
    label       = int(proba_phish >= threshold)
    verdict     = 'PHISHING' if label == 1 else 'LEGITIMATE'

    return {
        'url':               url,
        'verdict':           verdict,
        'phish_probability': round(float(proba_phish), 4),
        'label':             label,
        'threshold':         round(threshold, 3),
    }

# ── Test on real URLs ────────────────────────────────────────────────────────
test_urls = [
    ('https://www.google.com',                               'LEGIT'),
    ('https://www.github.com/Matheesha-Malshan',                         'LEGIT'),
    ('https://www.stackoverflow.com/questions/123456',           'LEGIT'),
    ('http://www.bbc.co/news',                            'LEGIT'),
    ('http://192.168.1.1/login/verify?user=admin',           'PHISH'),
    ('http://www.paypal-secure-login.xyz/account/verify',        'PHISH'),
    ('http://www.amazon.com.phishing.tk/signin',             'PHISH'),
    ('https://www.secure-bankofamerica.verify-account.ml/login', 'PHISH'),
]

print(f'{"Expected":<8}  {"Got":<10}  {"Prob":>6}  URL')
print('-' * 80)
for url, expected in test_urls:
    r = predict_url(url)
    match = '✓' if (expected == 'LEGIT') == (r['verdict'] == 'LEGITIMATE') else '✗'
    print(f"{match} {expected:<8}  {r['verdict']:<10}  {r['phish_probability']:.3f}  {url[:55]}")

Expected  Got           Prob  URL
--------------------------------------------------------------------------------
✓ LEGIT     LEGITIMATE  0.029  https://www.google.com
✓ LEGIT     LEGITIMATE  0.002  https://www.github.com/Matheesha-Malshan
✓ LEGIT     LEGITIMATE  0.005  https://www.stackoverflow.com/questions/123456
✗ LEGIT     PHISHING    1.000  http://www.bbc.co/news
✓ PHISH     PHISHING    1.000  http://192.168.1.1/login/verify?user=admin
✓ PHISH     PHISHING    1.000  http://www.paypal-secure-login.xyz/account/verify
✓ PHISH     PHISHING    1.000  http://www.amazon.com.phishing.tk/signin
✓ PHISH     PHISHING    0.999  https://www.secure-bankofamerica.verify-account.ml/logi


In [15]:
# DEBUG — run this right after training
import pandas as pd

# 1. What do legit URLs look like in the NEW dataset?
print("=== LEGIT URL samples from training data ===")
legit_urls = df_work[df_work['label']==0]['url'].head(20).tolist()
for u in legit_urls:
    print(u)

print("\n=== PHISHING URL samples ===")
phish_urls = df_work[df_work['label']==1]['url'].head(10).tolist()
for u in phish_urls:
    print(u)

# 2. Feature means by class — the key diagnostic
print("\n=== Feature means: LEGIT(0) vs PHISHING(1) ===")
compare = X.join(y).groupby('label')[[
    'url_length', 'slash_path_ratio', 'n_path_dirs',
    'is_https', 'n_subdomains', 'is_known_legit_domain'
]].mean().round(4)
print(compare)

# 3. What features does the model actually use most?
importances = pd.Series(best_model.feature_importances_, index=FEATURE_NAMES)
print("\n=== Top 10 most important features ===")
print(importances.sort_values(ascending=False).head(10))

# 4. Show raw feature values for a URL that should be LEGIT
print("\n=== Features for github.com/user/repo ===")
f = extract_features('https://github.com/user/repo')
for k, v in f.items():
    print(f"  {k:<25} = {v}")

=== LEGIT URL samples from training data ===
https://www.google.com
https://www.youtube.com
https://www.facebook.com
https://www.baidu.com
https://www.wikipedia.org
https://www.reddit.com
https://www.yahoo.com
https://www.google.co.in
https://www.qq.com
https://www.amazon.com
https://www.taobao.com
https://www.twitter.com
https://www.tmall.com
https://www.google.co.jp
https://www.vk.com
https://www.live.com
https://www.instagram.com
https://www.sohu.com
https://www.sina.com.cn
https://www.jd.com

=== PHISHING URL samples ===
http://atualizacaodedados.online
http://webmasteradmin.ukit.me/
http://stcdxmt.bigperl.in/klxtv/apps/uk/
https://tubuh-syarikat.com/plugins/fields/files/
http://rolyborgesmd.com/exceword/excel.php?.rand=13InboxLight.aspx?n==
http://ongelezen-voda.000webhostapp.com/inloggen.html
http://www.valenzaceramic.com/home/webapps/e52c1/websrc
http://membership-issue.forteimpex.com/dk2mmm=/?resType=code&amp;failedBe=
http://membership-issue.forteimpex.com/dk2mmm=/?resType=cod

In [16]:
# How many legit URLs have www. vs not?
legit_urls = df_work[df_work['label']==0]['url']
has_www = legit_urls.str.contains('://www\.').sum()
no_www  = len(legit_urls) - has_www
print(f"Legit with www    : {has_www} ({has_www/len(legit_urls)*100:.1f}%)")
print(f"Legit without www : {no_www}  ({no_www/len(legit_urls)*100:.1f}%)")

# How many legit URLs have any path?
has_path = legit_urls.str.count('/').gt(2).sum()
print(f"Legit with path   : {has_path} ({has_path/len(legit_urls)*100:.1f}%)")

Legit with www    : 345042 (99.8%)
Legit without www : 696  (0.2%)
Legit with path   : 343238 (99.3%)
